# MFT Acceptance — full pipeline

**Run order:** top to bottom. Cells 1–2 define everything; Cell 3 (optional, ~9 min) recomputes the per-config Peff files; Cells 4–5 generate the RU weights and plots (these do **not** need Cell 3).

## Cell 1 — Setup: chip dataframe, library, geometry helpers

In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import math as m
import time
import os
from itertools import combinations
import re
from collections import Counter
from github import Github
from distinctipy import get_colors

rows = []

with open("chipPositionsnew.txt", "r") as file:
    for line in file:
        line = line.strip()

        if line == "":
            continue

        match = re.search(
            r"ChipID\s+(\d+)\s*:\s*(h\d+)-(d\d+)\s+(layer\d+)\s+(z\d+)-?(tr\d+)\s+Position:\s*x=([-+]?\d+\.\d+),\s*y=([-+]?\d+\.\d+),\s*z=([-+]?\d+\.\d+)",
            line
        )

        if match:
            chip_id = int(match.group(1))
            half = match.group(2)
            disk = match.group(3)
            layer = match.group(4)
            zone = match.group(5)
            tr = match.group(6)
            x = float(match.group(7))
            y = float(match.group(8))
            z = float(match.group(9))

            rows.append([chip_id, half, disk, layer, zone, tr, x, y, z])

Column_labels = ['ChipID', 'Half', 'Disk', 'Layer', 'Zone', 'Tr', 'X', 'Y', 'Z']

Chip_DF = pd.DataFrame(rows, columns=Column_labels)
Chip_DF.index = Chip_DF.index + 1

print("\n------------- Building chip dataframe -------------")
print(Chip_DF.head())

%run Library/XY_2D_CUSTOM_HIST.ipynb
%run Library/pd_DF.ipynb
%run Library/Retrieve_ChipID.ipynb

DimChip = [13, 30, 0]
dim1, dim2 = 400, 400
zIP = 0

def GetCoor(ChipID, DF=Chip_DF):
    X = (DF.at[ChipID, 'X'] + 0.65) * 10
    Y = (DF.at[ChipID, 'Y'] + 1.5) * 10
    Z = DF.at[ChipID, 'Z'] - zIP
    return [X, Y, Z]

def GetTrueCoor(ChipID, DF=Chip_DF):
    X = DF.at[ChipID, 'X'] + 0.65
    Y = DF.at[ChipID, 'Y'] + 1.5
    Z = DF.at[ChipID, 'Z'] - zIP
    return [X, Y, Z]

# mean z of each layer, used by the weight generator
Z_MEANS = {L: int(np.mean(BuildLayerDF(f'layer{L}')['Z'].values)) for L in range(10)}

## Cell 2 — Array builders & pseudo-efficiency functions

In [ ]:
def Generate_Active_Area_Arrays(layers):
    active_layer_area = [0]*10
    for layer in layers:
        idx = int(layer[5])
        DF = BuildLayerDF(layer)

        active_layer_area[idx] = np.zeros((dim1, dim2))

        for chip in DF['ChipID'].values:
            X,Y,Z = GetCoor(chip+1, DF)
            Rect = DrawRectangle(round(X+dim1/2),round(Y+dim2/2))
            active_layer_area[idx] = np.logical_or(active_layer_area[idx], Rect)
    return active_layer_area

def BuildDecommissionedArray(decommissioned_chips):
    decomissioned_array = [np.zeros((dim1,dim2))]*10
    
    for chip in decommissioned_chips:
        X, Y, Z = GetCoor(chip+1)
        Rect = DrawRectangle(round(X+dim1/2), round(Y+dim2/2))
        
        layer = ExtractLayer(chip)
        decomissioned_array[layer] = np.logical_or(decomissioned_array[layer], Rect)
        
    return decomissioned_array

def Build_MFT_Array():
    layers = ['layer0', 'layer1', 'layer2', 'layer3', 'layer4', 'layer5', 'layer6', 'layer7', 'layer8', 'layer9']
    
    MFT_array = [np.zeros((dim1,dim2))]*10
    
    for chip in Chip_DF['ChipID'].values:
        X, Y, Z = GetCoor(chip+1)
        Rect = DrawRectangle(round(X+dim1/2), round(Y+dim2/2))
        layer = ExtractLayer(chip)
        MFT_array[layer] = np.logical_or(MFT_array[layer], Rect)
        
    return MFT_array

def Build_Config_Array(Config, decomissioned_array, MFT_array=Build_MFT_Array()):
    Config_Array = [np.ones((dim1,dim2))]*10
    Config_Name = 'Config'
    for layer in Config:
        l = int(layer[5])
        Config_Name = Config_Name + '_' + layer[5]
        if np.sum(decomissioned_array[l])!=0:
            Config_Array[l] = np.logical_xor(MFT_array[l], decomissioned_array[l])
        else:
            Config_Array[l] = MFT_array[l]
            
    return Config_Array, Config_Name

def BuildOmegaProjections(chip):
    Omega_Projections = [np.zeros((dim1,dim2))]*10
    X, Y, Z = GetCoor(chip+1)
    
    for layer in range(0,10):
        DF = BuildLayerDF(f"layer{layer}")
        z_mean = int(np.mean(DF['Z'].values))
        
        xb, yb, t = PointOnLine(X, Y, Z, z_mean)
        xb += dim1/2
        yb += dim2/2
            
        Rect = DrawRectangle(round(xb),round(yb), t)   
        
        Omega_Projections[layer] = np.logical_or(Omega_Projections[layer], Rect)
    
    return Omega_Projections

def BuildOverlapResults(chip, config, Config_Array_reduced):
    Omega_Projections = [np.zeros((dim1,dim2))]*10
    Overlap_Array = [np.zeros((dim1,dim2))]*10
    Overlap_Results = [1]*10
    
    X, Y, Z = GetCoor(chip+1)
    
    layers = [int(layer[5]) for layer in config]
    
    for layer in layers:
        DF = BuildLayerDF(f"layer{layer}")
        z_mean = int(np.mean(DF['Z'].values))
        
        xb, yb, t = PointOnLine(X, Y, Z, z_mean)
        xb += dim1/2
        yb += dim2/2
            
        Rect = DrawRectangle(round(xb),round(yb), t)   
        
        Omega_Projections[layer] = np.logical_or(Omega_Projections[layer], Rect)
        
        Overlap_Array[layer] = np.logical_and(Config_Array_reduced[layer], Omega_Projections[layer])
        
        Overlap_Results[layer] = round(np.sum(Overlap_Array[layer])/np.sum(Omega_Projections[layer]), 3)
    
    Peff = round(100*np.prod(Overlap_Results), 2)
    Overlap_Results[9] =  Peff
    
    return Overlap_Results

# ============================================================
# NEW: fast versions — cache only the cheap geometry numbers per
# chip per layer (xb, yb, t), NOT the rasterized (400,400) array.
# Avoids the MemoryError while still skipping redundant BuildLayerDF
# and PointOnLine calls.
# ============================================================

def BuildChipProjectionParams(chip):
    Params = [None]*10
    X, Y, Z = GetCoor(chip+1)
    for layer in range(10):
        z_mean = Z_MEANS[layer]
        xb, yb, t = PointOnLine(X, Y, Z, z_mean)
        xb += dim1/2
        yb += dim2/2
        Params[layer] = (round(xb), round(yb), t)
    return Params

def BuildOverlapResults_Fast(chip_params, config, Config_Array_reduced):
    Overlap_Results = [1]*10
    layers = [int(layer[5]) for layer in config]
    for layer in layers:
        xb, yb, t = chip_params[layer]
        Rect = DrawRectangle(xb, yb, t)
        Overlap_Array = np.logical_and(Config_Array_reduced[layer], Rect)
        Overlap_Results[layer] = round(np.sum(Overlap_Array)/np.sum(Rect), 3)
    Peff = round(100*np.prod(Overlap_Results), 2)
    Overlap_Results[9] = Peff
    return Overlap_Results

## Cell 3 — Core Peff computation (optional, ~9 min → writes `Config_*.txt`)

In [ ]:
start_time0 = time.perf_counter()
layers = ['layer0', 'layer1', 'layer2', 'layer3', 'layer4', 'layer5', 'layer6', 'layer7', 'layer8', 'layer9']

DimChip = [13,30,0]
dim1, dim2 = 400, 400
zIP = 0   # matches the reference dataset

def GetCoor(ChipID, DF = Chip_DF):
    X = (DF.at[ChipID, 'X']+0.65)*10
    Y = (DF.at[ChipID, 'Y']+1.5)*10
    Z = DF.at[ChipID, 'Z']-zIP
    return [X,Y,Z]

def GetTrueCoor(ChipID, DF = Chip_DF):
    X = DF.at[ChipID, 'X']+0.65
    Y = DF.at[ChipID, 'Y']+1.5
    Z = DF.at[ChipID, 'Z']-zIP
    return [X,Y,Z]

MFT_array = Build_MFT_Array()

Dead_Map = [2,8,45,46,47,68,69,102,103,104,470,476,511,536,546,547,548,549,550,551]
decommissioned_chips = Dead_Map
decomissioned_array = BuildDecommissionedArray(decommissioned_chips)

# Precompute layer z-positions once
Z_MEANS = {L: int(np.mean(BuildLayerDF(f"layer{L}")['Z'].values)) for L in range(10)}

# Precompute cheap per-chip, per-layer geometry (NOT the arrays themselves)
t0 = time.perf_counter()
Chip_Params = {chip: BuildChipProjectionParams(chip) for chip in Chip_DF['ChipID'].values}
print("Precompute time:", round(time.perf_counter() - t0, 1), "seconds for", len(Chip_Params), "chips")

for NbL in [10,9,8,7,6,5]:
    Config_List = LayerCombinations(NbL)
    path_NbL = f'Dead_Map/Acceptance_Dataframes/z_{zIP}/{NbL}_Clusters/'
    CreateDirectory(path_NbL)
    CreateDirectory(f"Dead_Map/Acceptance_Dataframes/z_{zIP}/Weights/")

    Config_Names = []
    
    for config in Config_List:
        Config_Array, Config_Name = Build_Config_Array(config, decomissioned_array, MFT_array)
        Config_Names.append('.'.join(layer[5] for layer in config))
        
        Last_Layer = config[-1]
        DF_Last_Layer = BuildLayerDF(Last_Layer)
    
        Layer0_list, Layer1_list, Layer2_list, Layer3_list, Layer4_list = [],[],[],[],[]
        Layer5_list, Layer6_list, Layer7_list, Layer8_list, Peff_list = [],[],[],[],[]
        
        for chip in DF_Last_Layer['ChipID'].values:
            Layer0, Layer1, Layer2, Layer3, Layer4, Layer5, Layer6, Layer7, Layer8, Peff = BuildOverlapResults_Fast(
                Chip_Params[chip], config, Config_Array
            )
            Layer0_list.append(Layer0)
            Layer1_list.append(Layer1)
            Layer2_list.append(Layer2)
            Layer3_list.append(Layer3)
            Layer4_list.append(Layer4)
            Layer5_list.append(Layer5)
            Layer6_list.append(Layer6)
            Layer7_list.append(Layer7)
            Layer8_list.append(Layer8)
            Peff_list.append(Peff)

        d = {'ChipID':DF_Last_Layer['ChipID'].values, 'Half':DF_Last_Layer['Half'].values,
             'Layer0':Layer0_list, 'Layer1':Layer1_list, 'Layer2':Layer2_list, 'Layer3':Layer3_list, 
             'Layer4':Layer4_list, 'Layer5':Layer5_list, 'Layer6':Layer6_list, 'Layer7':Layer7_list, 
             'Layer8':Layer8_list, 'Peff(%)':Peff_list}
        df = pd.DataFrame(data=d)
        df.to_csv(f"{path_NbL}/{Config_Name}.txt", sep='\t', index=False)
        
print("--- %s minutes ---" % (round((time.perf_counter() - start_time0)/60,3)))

## Cell 4 — Generate RU weights (all 5 disks)

In [ ]:
# ============ RU-WEIGHT GENERATOR (reconstructs the lost writer) ============
#  A Read-out Unit (RU) is one phi/radial zone of ~16 chips on ONE layer.
#  weight = geometric share of tracks (seeded on the config's LAST layer) whose
#  straight IP-line passes back through THIS zone on the RU's own layer.
#    * 0 if the config does not use the RU's own layer
#    * depends only on (RU, last layer); the 4 zones of a (half,disk,face) sum to ~100%
#  -> flat horizontal lines. Validated against the reference data (disks 0-1) to ~1-2%.

_zonefrac_cache = {}

def _zone_area(half, disk, face, zone):
    layer = disk*2 + face
    A = np.zeros((dim1, dim2))
    for c in Get_RU_chips(f"h{half}-d{disk} layer{layer} z{zone}", "chipPositionsnew.txt"):
        X, Y, Z = GetCoor(c+1)
        A = np.logical_or(A, DrawRectangle(round(X+dim1/2), round(Y+dim2/2)))
    return A

def zonefracs(half, disk, face, last):
    key = (half, disk, face, last)
    if key in _zonefrac_cache:
        return _zonefrac_cache[key]
    Lru = disk*2 + face
    areas = {zn: _zone_area(half, disk, face, zn) for zn in range(4)}
    if all(np.sum(areas[zn]) == 0 for zn in range(4)):
        _zonefrac_cache[key] = None
        return None
    ov = {zn: 0.0 for zn in range(4)}
    L_DF = BuildLayerDF(f"layer{last}")
    L_DF = L_DF[L_DF["Half"] == f"h{half}"]
    for chip in L_DF["ChipID"].values:
        X, Y, Z = GetCoor(chip+1)
        xb, yb, t = PointOnLine(X, Y, Z, Z_MEANS[Lru])
        Rect = DrawRectangle(round(xb+dim1/2), round(yb+dim2/2), t)
        s = np.sum(Rect)
        if s == 0:
            continue
        for zn in range(4):
            ov[zn] += np.sum(np.logical_and(areas[zn], Rect)) / s
    tot = sum(ov.values())
    res = {zn: (100*ov[zn]/tot if tot > 0 else 0.0) for zn in range(4)}
    _zonefrac_cache[key] = res
    return res

def Build_Disk_RUs(disk):
    labels, chips = [], []
    for half in [0, 1]:
        for layer in range(10):
            face = layer % 2
            for zone in [0, 1, 2, 3]:
                name = f"h{half}-d{disk} f{face} z{zone}"
                ch = Get_RU_chips(f"h{half}-d{disk} layer{layer} z{zone}", "chipPositionsnew.txt")
                if len(ch) > 0 and name not in labels:
                    labels.append(name); chips.append(ch)
    return labels, chips

def Generate_RU_Weights_Geometric(disk, NbL_list=(5, 6, 7, 8, 9, 10)):
    RU_List, _ = Build_Disk_RUs(disk)
    for NbL in NbL_list:
        configs = LayerCombinations_C(NbL)
        for RU in RU_List:
            hd, f_str, z_str = RU.split()
            half, face, zone = int(hd[1]), int(f_str[1]), int(z_str[1])
            Lru = disk*2 + face
            outdir = f"Acceptance_Dataframes/RU_Weights/{RU}/z_0/Weights"
            CreateDirectory(outdir)
            with open(f"{outdir}/Weights_NbL_{NbL}.txt", "w") as fout:
                fout.write("Configurations\tWeights(%)\n")
                for cfg in configs:
                    cfg_layers = [int(l[5]) for l in cfg]
                    label = ".".join(str(x) for x in cfg_layers)
                    if Lru not in cfg_layers:
                        w = 0.0
                    else:
                        zf = zonefracs(half, disk, face, cfg_layers[-1])
                        w = round(zf[zone], 2) if zf else 0.0
                    fout.write(f"{label}\t{w}\n")
    print(f"disk {disk}: wrote weights for {len(RU_List)} RUs")

for disk in [0, 1, 2, 3, 4]:
    Generate_RU_Weights_Geometric(disk)
print("All weights generated.")

## Cell 4b — Overlay the ground-truth reference weights (exact where they exist)

The inherited repository preserved the real `Weights_NbL_*.txt` for **disk 0 (h0) and part of disk 1** —
these are kept in `RU_Weights_GroundTruth/`. This cell copies them **over** the generated files,
so those RUs are bit-identical to the reference results; all other RUs keep the validated ~98% geometric
reconstruction (the original files for disks 2–4 were never saved anywhere and cannot be recovered).

In [ ]:
# ============ OVERLAY GROUND TRUTH (exact reference values where available) ============
import shutil

def Overlay_Ground_Truth(src_root="RU_Weights_GroundTruth",
                         dst_root="Acceptance_Dataframes/RU_Weights"):
    n = 0
    if not os.path.isdir(src_root):
        print(f"No {src_root}/ folder found - skipping overlay.")
        return 0
    for RU in os.listdir(src_root):
        wdir = os.path.join(src_root, RU, "z_0", "Weights")
        if not os.path.isdir(wdir):
            continue
        dst = os.path.join(dst_root, RU, "z_0", "Weights")
        CreateDirectory(dst)
        for fn in os.listdir(wdir):
            shutil.copy(os.path.join(wdir, fn), os.path.join(dst, fn))
            n += 1
        print(f"exact (reference): {RU}")
    print(f"
Overlaid {n} ground-truth files. These RUs now match the reference bit-for-bit;")
    print("all remaining RUs use the geometric reconstruction (~98%, validated on disks 0-1).")
    return n

Overlay_Ground_Truth()

## Cell 5 — Plot RU-weight vs φ → `Weights_D0..D4.png`

In [ ]:
# ============ PLOT RU-WEIGHT vs PHI  ->  Weights_D0..D4.png ============
def calculate_average_weight(fp):
    ws = [float(l.split('	')[1]) for l in open(fp).read().splitlines()[1:]]
    ws = [w for w in ws if w == w and w != 0]      # drop NaN and zero
    return sum(ws)/len(ws) if ws else float('nan')

def Plot_RU_Weights_Phi(disk):
    RU_List, RU_Chips = Build_Disk_RUs(disk)
    Missing_RU = []
    fig, axs = plt.subplots(3, 2, figsize=(12, 12), constrained_layout=True)
    cmap = plt.cm.tab20(np.linspace(0, 1, max(len(RU_List), 1)))
    for i, NbL in enumerate([10, 9, 8, 7, 6, 5]):
        r, c = divmod(i, 2)
        axs[r, c].set_title(f"{NbL} layers"); axs[r, c].set_xlabel("Phi")
        axs[r, c].set_ylabel("Weight (%)"); axs[r, c].set_ylim(0, 70); axs[r, c].grid()
        for idx, (RU, chips) in enumerate(zip(RU_List, RU_Chips)):
            fp = f"Acceptance_Dataframes/RU_Weights/{RU}/z_0/Weights/Weights_NbL_{NbL}.txt"
            if os.path.isfile(fp):
                print(f'Data for {RU}!!!')
                w = calculate_average_weight(fp)
                phis = [ChangeToEtaPhi2(ch)[1] for ch in chips]
                order = np.argsort(phis)
                axs[r, c].plot(np.array(phis)[order], [w]*len(phis), label=RU, color=cmap[idx])
            else:
                print(f'No data for {RU}!!!')
                Missing_RU.append(RU)
        axs[r, c].legend(loc=4, bbox_to_anchor=(1.28, 0), fontsize=6)
    CreateDirectory("Acceptance_Results/z_0")
    fig.savefig(f"Acceptance_Results/z_0/Weights_D{disk}.png", dpi=90)
    plt.show()
    if len(Missing_RU) > 0:
        return Missing_RU

for disk in [0, 1, 2, 3, 4]:
    Plot_RU_Weights_Phi(disk)
print("All 5 plots saved to Acceptance_Results/z_0/")